# 03 · Feature engineering → SageMaker Feature Store
**AAI-540 · Group 4 · Criteo CTR**

### Feature group design
| Feature group | Record id / event time | Features | Why separate |
|---|---|---|---|
| `criteo-ctr-numeric-fg` | `record_id` / `event_time` | `day_index`, `I1–I13` (signed log1p, imputed 0), `I1_missing–I13_missing` | Dense numeric signals; monitored for distribution drift |
| `criteo-ctr-categorical-fg` | `record_id` / `event_time` | `C1_hash–C26_hash` (hex id → fixed 2²⁰ space; 0 = missing) | High-cardinality ids; monitored for unseen-level rate |
| `criteo-ctr-label-fg` | `record_id` / `event_time` | `label` | Isolated so no inference path can read the target |

**Only stateless transforms are stored here.** Anything that learns from data (rare-level collapsing,
frequency encoding) is fitted on the training split in notebook 04; storing it here would leak validation, test and
production statistics into training.

Offline store only (S3 + Glue/Athena) to control cost; the online store is enabled later if the real-time
endpoint is built. Ingestion writes ≈ 3 × 1 M records; set `CRITEO_FS_INGEST_LIMIT` for a quick dry run.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import sagemaker
import pandas as pd
from botocore.exceptions import ClientError

from criteo_ctr import config as C
from criteo_ctr import features as F
from criteo_ctr.athena import Athena
from criteo_ctr.io_utils import Store, s3_uri, lower_columns, canonical_columns, default_data_dir

sess = sagemaker.Session()
boto_sess = sess.boto_session
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)
athena = Athena(boto_sess, s3_uri(bucket, C.ATHENA_RESULTS_PREFIX) + "/", database=C.ATHENA_DATABASE)

print(f"region={region}\nbucket={bucket}\nprefix={C.PREFIX}\nathena db={C.ATHENA_DATABASE}")

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup

sample = store.read_parquet_prefix(C.SAMPLE_PREFIX)
sample = sample.astype({**{c: "Int64" for c in C.NUM_COLS}, **{c: "string" for c in C.CAT_COLS},
                        C.LABEL: "int64", C.DAY_INDEX: "int64", C.RECORD_ID: "int64", C.EVENT_TIME: "float64"})
sample = sample.sort_values(C.RECORD_ID).reset_index(drop=True)

LIMIT = int(os.environ.get("CRITEO_FS_INGEST_LIMIT", "0")) or None
if LIMIT:
    sample = sample.head(LIMIT)
    print(f"DRY RUN: limited to {LIMIT:,} records")
print(f"{len(sample):,} records to engineer and ingest")

## 1 · Engineer stateless features

In [ ]:
t0 = time.time()
fg_frames = F.build_feature_group_frames(sample)
print(f"engineered in {time.time()-t0:.1f}s")
for name, df in fg_frames.items():
    assert df[C.RECORD_ID].is_unique and not df.isna().any().any(), name
    print(f"{name:28s} {df.shape[0]:>10,} rows x {df.shape[1]:>3} cols | dtypes: {sorted(set(map(str, df.dtypes)))}")
fg_frames[C.FG_NUMERIC].head(3)

In [ ]:
# Sanity checks on the engineered values before they become the system of record.
num_fg, cat_fg = fg_frames[C.FG_NUMERIC], fg_frames[C.FG_CATEGORICAL]
for c in C.NUM_COLS:
    assert (num_fg[f"{c}_missing"] == sample[c].isna().astype(int)).all(), c
for c in C.CAT_COLS:
    h = cat_fg[f"{c}_hash"]
    assert ((h == C.MISSING_ID) == sample[c].isna().to_numpy()).all(), c
    assert h.between(0, C.HASH_BUCKETS - 1).all(), c
print("feature sanity checks passed")

## 2 · Create the feature groups (idempotent)

In [ ]:
DESCRIPTIONS = {
    C.FG_NUMERIC: "Criteo CTR - I1-I13 signed log1p (missing->0) plus missing-indicator flags, and day_index",
    C.FG_CATEGORICAL: "Criteo CTR - C1-C26 hashed into 2^20 buckets; 0=missing, 1 reserved for rare/unseen",
    C.FG_LABEL: "Criteo CTR - click label (1=click). Kept separate from inference features by design",
}
OFFLINE_URI = s3_uri(bucket, C.FEATURE_STORE_OFFLINE_PREFIX)


def ensure_feature_group(name, frame):
    fg = FeatureGroup(name=name, sagemaker_session=sess)
    try:
        status = fg.describe()["FeatureGroupStatus"]
        print(f"{name}: already exists ({status})")
    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceNotFound":
            raise
        fg.load_feature_definitions(data_frame=frame)
        fg.create(s3_uri=OFFLINE_URI, record_identifier_name=C.RECORD_ID,
                  event_time_feature_name=C.EVENT_TIME, role_arn=role,
                  enable_online_store=False, description=DESCRIPTIONS[name],
                  tags=[{"Key": "project", "Value": "criteo-ctr"}, {"Key": "team", "Value": "aai540-group4"}])
        print(f"{name}: creating...")
    while True:
        d = fg.describe()
        status = d["FeatureGroupStatus"]
        if status != "Creating":
            break
        time.sleep(10)
    if status != "Created":
        raise RuntimeError(f"{name}: {status} - {d.get('FailureReason')}")
    return fg


feature_groups = {name: ensure_feature_group(name, frame) for name, frame in fg_frames.items()}
pd.DataFrame([{"feature_group": n, "status": fg.describe()["FeatureGroupStatus"],
               "n_features": len(fg.describe()["FeatureDefinitions"]),
               "offline_table": fg.describe()["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]}
              for n, fg in feature_groups.items()])

## 3 · Ingest

In [ ]:
ingest_report = {}
for name, frame in fg_frames.items():
    t0 = time.time()
    feature_groups[name].ingest(data_frame=frame, max_workers=8, max_processes=4, wait=True)
    secs = time.time() - t0
    ingest_report[name] = {"records": int(len(frame)), "seconds": round(secs, 1),
                           "records_per_sec": round(len(frame) / secs, 1)}
    print(f"{name}: {len(frame):,} records in {secs:,.0f}s ({len(frame)/secs:,.0f}/s)")

## 4 · Wait for the offline store, then verify through Athena
The offline store is populated asynchronously (typically 5–15 minutes after ingestion). Verification counts
**distinct** `record_id`s so a re-run that re-ingests records does not produce a false pass.

In [ ]:
def offline_distinct(fg):
    q = fg.athena_query()
    try:
        return int(athena.query(f'SELECT count(DISTINCT record_id) AS n FROM "{q.table_name}"',
                                database=q.database).n[0])
    except RuntimeError:
        return 0   # table has no data files yet


expected = len(sample)
deadline = time.time() + 45 * 60
pending = set(feature_groups)
while pending and time.time() < deadline:
    for name in sorted(pending):
        n = offline_distinct(feature_groups[name])
        print(f"{dt.datetime.now():%H:%M:%S} {name}: {n:,}/{expected:,}")
        if n >= expected:
            pending.discard(name)
    if pending:
        time.sleep(60)
assert not pending, f"offline store not complete for {pending} - re-run this cell"
print("offline store complete for all feature groups")

In [ ]:
# Spot-check: the same record read back from each offline table
rid = int(sample[C.RECORD_ID].iloc[len(sample) // 2])
for name, fg in feature_groups.items():
    q = fg.athena_query()
    row = athena.query(f'SELECT * FROM "{q.table_name}" WHERE record_id = {rid} LIMIT 1', database=q.database)
    print(name); display(row.iloc[:, :10])

In [ ]:
fs_meta = {"created_utc": dt.datetime.utcnow().isoformat() + "Z", "offline_store_uri": OFFLINE_URI,
           "records": expected, "ingest": ingest_report, "feature_groups": {}}
for name, fg in feature_groups.items():
    d, q = fg.describe(), fg.athena_query()
    fs_meta["feature_groups"][name] = {
        "arn": d["FeatureGroupArn"], "athena_database": q.database, "athena_table": q.table_name,
        "record_identifier": C.RECORD_ID, "event_time_feature": C.EVENT_TIME,
        "features": [f["FeatureName"] for f in d["FeatureDefinitions"]]}
print(store.put_json(fs_meta, f"{C.ARTIFACTS_PREFIX}/feature_store_metadata.json"))